This one will go to the top of the leaderboard

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression

In [3]:
import os
from functools import reduce

# All prediction folders
folders = [
    'PS-s6e4-07/',
    'PS-s6e4-74/',
    'PS-s6e4-85/',
    'PS-s6e4-89/'   
]

base_path = ""   # set if needed

sample_sub = pd.read_csv('playground-series-s6e4/sample_submission.csv')

In [4]:
all_dfs = []

for folder in folders:
    files = os.listdir(base_path + folder)
    
    for file in files:
        if file.endswith('.csv'):
            path = base_path + folder + file
            
            df = pd.read_csv(path)
            
            # Unique column name (VERY IMPORTANT)
            col_name = file.replace('.csv', '')
            
            df = df.rename(columns={'Irrigation_Need': col_name})
            
            all_dfs.append(df)

print(f"Total models loaded: {len(all_dfs)}")

Total models loaded: 42


In [5]:
dfs = reduce(lambda left, right: pd.merge(left, right, on='id'), all_dfs)

print("Shape:", dfs.shape)
dfs.head()

Shape: (270000, 43)


,id,0.97891,0.97902,0.97922,0.97956,0.97971.a,0.97971.b,0.97971.c,0.97971,0.97971.d,...,Eos5 - 0.98066,Eos6 - 0.98066,Eos7 - 0.98067,Eos8 - 0.98085,Top - 0.98088,0.98011.a,0.98011.b,0.98011.c,0.98011_y,0.98026
0,630000,Low,Low,Low,Low,Low,Low,Low,Low,Low,...,Low,Low,Low,Low,Low,Low,Low,Low,Low,Low
1,630001,Low,Low,Low,Low,Low,Low,Low,Low,Low,...,Low,Low,Low,Low,Low,Low,Low,Low,Low,Low
2,630002,Low,Low,Low,Low,Low,Low,Low,Low,Low,...,Low,Low,Low,Low,Low,Low,Low,Low,Low,Low
3,630003,Low,Low,Low,Low,Low,Low,Low,Low,Low,...,Low,Low,Low,Low,Low,Low,Low,Low,Low,Low
4,630004,Low,Low,Low,Low,Low,Low,Low,Low,Low,...,Low,Low,Low,Low,Low,Low,Low,Low,Low,Low


In [ ]:
def majority_vote(row):
    return row.mode()[0]

dfs['final_pred'] = dfs.drop(columns=['id']).apply(majority_vote, axis=1)

submission = sample_sub.copy()
submission['Irrigation_Need'] = dfs['final_pred']

submission.to_csv('submission_majority.csv', index=False)
submission.head()

In [ ]:
# Assign weights based on LB score (adjust manually)
weights = {}

for col in dfs.columns:
    if col != 'id':
        if '0.9807' in col or '0.9808' in col:
            weights[col] = 1.5
        elif '0.980' in col:
            weights[col] = 1.3
        else:
            weights[col] = 1.0

def weighted_vote(row):
    vote_score = {'Low':0, 'Medium':0, 'High':0}
    
    for col in weights:
        vote_score[row[col]] += weights[col]
    
    return max(vote_score, key=vote_score.get)

dfs['weighted_pred'] = dfs.apply(weighted_vote, axis=1)

submission['Irrigation_Need'] = dfs['weighted_pred']
submission.to_csv('submission_weighted.csv', index=False)

In [ ]:
# Keep only strong models
strong_cols = [c for c in dfs.columns if any(x in c for x in ['0.980', '0.981'])]

dfs_strong = dfs[['id'] + strong_cols]

print("Using models:", len(strong_cols))

dfs_strong['final'] = dfs_strong.drop(columns=['id']).mode(axis=1)[0]

submission['Irrigation_Need'] = dfs_strong['final']
submission.to_csv('submission_strong_only.csv', index=False)